In [1]:
import torch

In [ ]:
https://www.sketchengine.eu/opus-parallel-corpora/: i ofund this intrestng


In [2]:
import argparse
import ast
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer
from transformers.models.marian import MarianMTModel

In [3]:
class TextDataset(Dataset):
    def __init__(self, tokenizer, original_data_path=None, text_data_list=None):
        self.tokenizer = tokenizer
        if original_data_path:
            self.df = pd.read_csv(original_data_path)
            self.text_data_list = self.df['text'].tolist()
            self.text_num_list = [1] * len(self.text_data_list)
        else:
            self.text_data_list = text_data_list
            self.text_num_list = [1] * len(text_data_list)
    
    def __len__(self):
        return len(self.text_data_list)
    
    def __getitem__(self, idx):
        return self.text_data_list[idx]
    
    def collate_fn(self, batch):
        return self.tokenizer(batch, return_tensors='pt', padding=True, truncation=True)

In [4]:
class BackTranslation:
    def __init__(self, lang="de"):
        self.lang = lang
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        try:
            print("USING:", self.device)
            self.en_lang_tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}")
            self.lang_en_tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en")
            self.en_lang_translator = MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}").to(self.device)
            self.lang_en_translator = MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en").to(self.device)
        except Exception as e:
            print(f"Warning: Could not load models for language {lang}: {str(e)}")
            self.en_lang_tokenizer = None

    def do_back_translation(self, original_data_path, batch_size, temperature, **generate_kwargs):
        if not self.en_lang_tokenizer:  # Skip if model loading failed
            return None, None
            
        assert len(temperature) <= 2
        temp1, temp2 = (temperature[0], temperature[0]) if len(temperature) == 1 else temperature
        pandas_text_dataset = TextDataset(self.en_lang_tokenizer, original_data_path=original_data_path)
        dataloader = DataLoader(
            pandas_text_dataset, shuffle=False, drop_last=False, num_workers=4, 
            batch_size=batch_size, collate_fn=pandas_text_dataset.collate_fn
        )
        text_num_list = pandas_text_dataset.text_num_list
        lang_out_list = []
        for batch in tqdm(dataloader, desc=f"Translating to {self.lang}"):
            lang_out = self.en_lang_translator.generate(**batch.to(self.device), temperature=temp1, **generate_kwargs)
            for out in lang_out:
                lang_out_list.append(self.en_lang_tokenizer.decode(out).replace("<pad>", "").replace("</s>", "").strip())
        
        lang_text_dataset = TextDataset(self.lang_en_tokenizer, text_data_list=lang_out_list)
        dataloader = DataLoader(
            lang_text_dataset, shuffle=False, drop_last=False, num_workers=4, 
            batch_size=batch_size, collate_fn=lang_text_dataset.collate_fn
        )
        en_out_list = []
        for batch in tqdm(dataloader, desc=f"Translating back from {self.lang}"):
            en_out = self.lang_en_translator.generate(**batch.to(self.device), temperature=temp2, **generate_kwargs)
            for out in en_out:
                en_out_list.append(self.lang_en_tokenizer.decode(out).replace("<pad>", "").replace("</s>", "").strip())
        
        text_augment_list = []
        start = 0
        for text_num in text_num_list:
            text_augment_list.append(en_out_list[start : start + text_num])
            start += text_num
            
        return lang_out_list, text_augment_list

In [5]:
def test_backtranslation():
    sample_data = pd.DataFrame({
        'text': ["The cat is on the mat", "Dogs love to play fetch"]
    })
    os.makedirs("dataset", exist_ok=True)
    sample_data.to_csv("dataset/dataset_train.csv", index=False)
    bt = BackTranslation(lang="de")
    bt.do_back_translation(
        original_data_path="dataset/dataset_train.csv",
        out_data_path="dataset/dataset_aug_train.csv",
        batch_size=256,
        temperature=[1.0],
        num_beams=5,
        do_sample=True
    )
    result_df = pd.read_csv("dataset/dataset_aug_train.csv")
    print("\nOriginal and Augmented Texts:")
    print(result_df[['text', 'text_augment']])
    with open("dataset/dataset_aug_train.de.txt", "r") as f:
        print("\nIntermediate German Translation:")
        print(f.read())

In [6]:
# import sys

# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--lang", type=str, default="de")
#     parser.add_argument("--temperature", nargs="+", type=float, default=[1.0])
#     parser.add_argument("--seed", type=int, default=42)
#     parser.add_argument("--bsz", type=int, default=128)
#     parser.add_argument("--num-beams", type=int, default=5)
    
#     # Parse args and ignore unknown ones (like Jupyter's -f argument)
#     args, unknown = parser.parse_known_args()
    
#     # Set random seed if needed
#     torch.manual_seed(args.seed)
    
#     # Run the test
#     test_backtranslation()
    
#     # Uncomment this section to run the original main loop instead
#     """
#     backtranslation = BackTranslation(lang=args.lang)
#     for data_split in ["train", "valid"]:
#         inp_data_path = f"dataset/dataset_{data_split}.csv"
#         out_data_path = f"dataset/dataset_aug_{data_split}.csv"
#         if not os.path.exists(os.path.dirname(out_data_path)):
#             os.makedirs(os.path.dirname(out_data_path))
#         backtranslation.do_back_translation(
#             inp_data_path, 
#             out_data_path, 
#             batch_size=args.bsz, 
#             num_beams=args.num_beams, 
#             temperature=args.temperature, 
#             do_sample=True
#         )
#     """

    

In [7]:
def run_multi_language_pipeline():
    languages = ["af", "sq", "ar", "hy", "eu", "bg", "bn", "ca", "zh", "hr", "cs", "da", "nl", "et", "fi", "fr", "gl", "ka", "de", "el", "gu", "ht", "he", "hi", "hu", "is", "id", "ga", "it", "ja",
                 "kn", "kk", "km", "ko", "lv", "lt", "mk", "ms", "ml", "mt", "mr", "ne", "no", "fa", "pl", "pt", "pa", "ro", "ru", "sr", "sk", "sl", "es", "sw", "sv", "ta", "te", "th", "tr", "uk", "ur", "vi", "cy", "yi", "zu"]   
    # Input and output paths
    original_data_path = "dataset/train.csv"
    output_path = "dataset/dataset_aug_train_all_1.csv"
    print("FLAG")
    # Parameters
    batch_size = 128
    temperature = [1.0]
    num_beams = 5
    

    # Load the original dataset
    original_df = pd.read_csv(original_data_path)
    print(len(original_df))
    # Process each language and store results
    for lang in languages:
        print(f"\nProcessing language: {lang}")
        
        bt = BackTranslation(lang=lang)
        intermediate_texts, augmented_texts = bt.do_back_translation(
            original_data_path=original_data_path,
            batch_size=batch_size,
            temperature=temperature,
            num_beams=num_beams,
            do_sample=True
        )
        
        if intermediate_texts and augmented_texts:
            # Add columns for this language
            original_df[f'intermediate_{lang}'] = intermediate_texts
            original_df[f'augment_{lang}'] = augmented_texts
        else:
            print(f"Skipping {lang} due to model loading failure")
            original_df[f'intermediate_{lang}'] = None
            original_df[f'augment_{lang}'] = None

    # Save all results to a single CSV
    original_df.to_csv(output_path, index=False)
    print(f"\nResults saved to {output_path}")
    
    # Print the results
    print("\nFinal Results:")
    print(original_df)

In [ ]:
if __name__ == "__main__":
    # Set random seed
    torch.manual_seed(42)
    run_multi_language_pipeline()

FLAG
1534699

Processing language: af
USING: cuda


/home/mtech/torch/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Translating to af:   0%|                  | 19/11990 [01:47<17:01:55,  5.12s/it]